# Experimental Medium-Range-Equivalent SWE Benchmark on a Flat Torus

Building on the equations and forcing definitions in the [shorter SWE walkthrough](01_01_shallow_water_equation.ipynb), this notebook uses a fifteen-day-equivalent rollout to test long-horizon stochastic emulation. It compares a rotational, dissipation-linked stochastic forecast with an unforced control restarted from the same briefly adjusted random-PV state.

The domain is a **flat, doubly periodic torus**, not a regional map or a sphere. It is useful for idealized synoptic vortices, jets, Rossby-like dynamics, and stochastic-forcing experiments, but it has no poles, equator, coastlines, or geographic topography. Fields leaving one edge re-enter through the opposite edge, so periodic recirculation is part of the experiment. The `periodic_beta` Coriolis profile is locally beta-plane-like near the domain centre; its gradient necessarily reverses elsewhere to remain periodic.

Atmospherically, $h$ represents one equivalent-depth, geopotential-like layer mode rather than literal atmospheric depth. Moisture, thermodynamics, and vertical structure are absent.

The default model scales are

$$c=\sqrt{gH}\simeq3.13,\qquad f_0=c/8\simeq0.392,\qquad L_D=c/f_0=8.$$

For interpretation only, taking $f_0=10^{-4}\,\mathrm{s}^{-1}$ and $c=30\,\mathrm{m\,s}^{-1}$ maps one model time unit to about 1.09 hours and one length unit to about 37.5 km. The $64\times64$ domain is then about 2400 km across and the deformation radius is about 300 km. These dimensional values are not hard-coded into the simulator.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

from autosim.experimental.simulations import ShallowWater2D
from autosim.utils import plot_spatiotemporal_video

g = 9.81
h_mean = 1.0
model_wave_speed = (g * h_mean) ** 0.5
model_f0 = model_wave_speed / 8.0
physical_f0 = 1e-4  # s^-1: illustrative midlatitude value
physical_wave_speed = 30.0  # m s^-1: illustrative equivalent mode
time_unit_seconds = model_f0 / physical_f0
time_unit_hours = time_unit_seconds / 3600.0
length_unit_metres = physical_wave_speed * time_unit_seconds / model_wave_speed
length_unit_km = length_unit_metres / 1000.0
velocity_unit = length_unit_metres / time_unit_seconds
equivalent_depth_metres = physical_wave_speed**2 / 9.81
model_beta = 0.5 * model_f0 / 64.0
physical_beta = model_beta / (length_unit_metres * time_unit_seconds)
inertial_period_days = 2.0 * torch.pi / model_f0 * time_unit_hours / 24.0
gravity_crossing_days = 64.0 / model_wave_speed * time_unit_hours / 24.0

print(f"1 model time unit ~= {time_unit_hours:.2f} hours")
print(f"1 model length unit ~= {length_unit_km:.1f} km")
print(f"domain width ~= {64 * length_unit_km:.0f} km")
print(f"deformation radius ~= {8 * length_unit_km:.0f} km")
print(f"equivalent depth ~= {equivalent_depth_metres:.0f} m")
print(f"local beta ~= {physical_beta:.2e} m^-1 s^-1")
print(f"inertial period ~= {inertial_period_days:.2f} days")
print(f"gravity-wave crossing time ~= {gravity_crossing_days:.2f} days")


## Initial-condition choices

The simulator provides three generated families. `random` retains the original mixed, jet-like state; `balanced_random_pv` produces distributed eddies at a controlled spectral scale; and `balanced_double_jet` isolates perturbed shear-flow dynamics. All are periodic and approximately balanced, while `restart` reuses any valid state rather than defining a fourth physical family. On this mapped domain, sampling Fourier modes 2--6 spans wavelengths of about 1200--400 km: within-domain synoptic variation, not global planetary structure.


In [ ]:
grid_size = 64  # Use 128 for better scale separation at higher cost.
domain_size = 64.0
fundamental_wavenumber = float(2.0 * torch.pi / domain_size)

physics = {
    "return_timeseries": True,
    "log_level": "warning",
    "nx": grid_size,
    "ny": grid_size,
    "Lx": domain_size,
    "Ly": domain_size,
    "cfl": 0.12,
    "g": g,
    "h_mean": h_mean,
    "nu": 5e-4,
    "drag": 2e-3,
    "coriolis_mode": "periodic_beta",
    "dtype": torch.float32,
}

initial_specs = {
    "original random": {"initial_condition": "random", "amp": 0.09},
    "balanced random PV": {
        "initial_condition": "balanced_random_pv",
        "amp": 0.7,
        "initial_wavenumber": 4.0 * fundamental_wavenumber,
        "initial_bandwidth": 1.5 * fundamental_wavenumber,
    },
    "balanced double jet": {
        "initial_condition": "balanced_double_jet",
        "amp": 0.7,
    },
}
initial_states = {}
for label, spec in initial_specs.items():
    options = {key: value for key, value in spec.items() if key != "amp"}
    initial_simulator = ShallowWater2D(
        **physics,
        **options,
        T=0.0,
        dt_save=1.0,
        parameters_range={"amp": (spec["amp"], spec["amp"])},
    )
    initial_states[label] = initial_simulator.forward_samples_spatiotemporal(
        n=1, random_seed=7
    )["data"][0, 0]

height_limit = max(
    float((state[..., 0] - h_mean).abs().max())
    for state in initial_states.values()
)
speed_limit = max(
    float(torch.linalg.vector_norm(state[..., 1:], dim=-1).max())
    for state in initial_states.values()
)
stride = 4
sampled_speeds = torch.cat(
    [
        torch.linalg.vector_norm(state[::stride, ::stride, 1:], dim=-1).flatten()
        for state in initial_states.values()
    ]
)
arrow_reference_speed = max(float(torch.quantile(sampled_speeds, 0.9)), 1e-8)
coordinates = torch.arange(0, grid_size, stride)
quiver_x, quiver_y = torch.meshgrid(coordinates, coordinates, indexing="xy")

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for column, (label, state) in enumerate(initial_states.items()):
    height_image = axes[0, column].imshow(
        (state[..., 0] - h_mean).T,
        origin="lower",
        cmap="RdBu_r",
        vmin=-height_limit,
        vmax=height_limit,
    )
    velocity = state[..., 1:]
    speed_image = axes[1, column].imshow(
        torch.linalg.vector_norm(velocity, dim=-1).T,
        origin="lower",
        cmap="magma",
        vmin=0.0,
        vmax=speed_limit,
    )
    axes[1, column].quiver(
        quiver_x,
        quiver_y,
        velocity[::stride, ::stride, 0].T,
        velocity[::stride, ::stride, 1].T,
        color="white",
        edgecolor="black",
        linewidth=0.3,
        pivot="mid",
        scale=arrow_reference_speed / (0.6 * stride),
        scale_units="xy",
        angles="xy",
    )
    axes[0, column].set_title(label)
    for row in range(2):
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])
axes[0, 0].set_ylabel(r"height anomaly $h-H$")
axes[1, 0].set_ylabel("speed and velocity")
fig.colorbar(height_image, ax=axes[0].tolist(), shrink=0.8)
fig.colorbar(speed_image, ax=axes[1].tolist(), shrink=0.8)
plt.show()


## Experiment design

The forecast uses `T=330`, `dt_save=6`, and `skip_nt=0`, giving 56 states over approximately fifteen interpreted days. A snapshot interval is about 6.5 hours, close to a conventional six-hour analysis cadence; adaptive CFL steps remain much shorter. The first five days emphasize paired response and predictability. At later leads, pointwise separation combines direct forcing with nonlinear amplification, so energy and distribution statistics become more informative.

An isotropic random-PV field centred on Fourier mode 4 undergoes a brief deterministic adjustment for 18 model time units, about one inertial period. Its reference-$f_0$ inversion is only approximately balanced under `periodic_beta`, not an exact steady state. Its wavelength is 16 model units: about 600 km or twice the deformation radius. Both forecasts then restart from the same saved tensor; unlike `skip_nt`, this guarantees a common post-adjustment state before forcing begins.

`vortical` forcing samples a streamfunction tendency $\psi_F$ and applies

$$F_h=0,\qquad (F_u,F_v)=(-\partial_y\psi_F,\partial_x\psi_F),\qquad \nabla\cdot\mathbf{F}_u=0.$$

This injects rotational momentum without directly perturbing height. Its Fourier-space streamfunction and Ornstein--Uhlenbeck time correlation follow the main ingredients of spectral stochastic backscatter, while the narrow homogeneous ring remains a benchmark simplification. The ring is centred on mode 8, a wavelength of about 300 km, and its correlation time is about 5.4 hours.

The fixed forcing floor is zero. Instead, the diffusion target is half the diagnosed viscosity and hyperviscosity loss; physical large-scale drag is deliberately excluded. This follows the energy-backscatter motivation without claiming to reproduce an operational scheme ([Berner et al., 2009](https://doi.org/10.1175/2008JAS2677.1)).

The mapped scales are plausible for idealized midlatitude flow, but the periodic geometry prevents a geographic forecast interpretation. The code reports the realized values. Spherical [PlanetSWE](https://polymathic-ai.org/the_well/datasets/planetswe/) additionally uses filtered ERA5 initial fields, topography, and deterministic moving height forcing; those geographically structured ingredients do not transfer directly to this flat torus.


In [ ]:
spinup_simulator = ShallowWater2D(
    **physics,
    T=18.0,
    dt_save=18.0,
    initial_condition="balanced_random_pv",
    initial_wavenumber=4.0 * fundamental_wavenumber,
    initial_bandwidth=1.5 * fundamental_wavenumber,
    parameters_range={"amp": (0.7, 0.7)},
)
spinup = spinup_simulator.forward_samples_spatiotemporal(n=1, random_seed=7)
restart_state = spinup["data"][0, -1].clone()

common = {
    **physics,
    "return_additional_input_fields": True,
    "return_energy_budget": True,
    "T": 330.0,
    "dt_save": 6.0,
    "skip_nt": 0,
    "initial_condition": "restart",
    "initial_state": restart_state,
    "forcing_wavenumber": 8.0 * fundamental_wavenumber,
    "forcing_bandwidth": fundamental_wavenumber,
    "forcing_correlation_time": 5.0,
    "forcing_backscatter_fraction": 0.5,
    "backscatter_include_drag": False,
    "parameters_range": {
        "amp": (0.7, 0.7),
        "forcing_energy_rate": (0.0, 0.0),
    },
}

runs = {}
for forcing_type in ("none", "vortical"):
    simulator = ShallowWater2D(forcing_type=forcing_type, **common)
    runs[forcing_type] = simulator.forward_samples_spatiotemporal(
        n=1, random_seed=7
    )

control = runs["none"]["data"][0]
forced = runs["vortical"]["data"][0]
torch.testing.assert_close(control[0], restart_state, rtol=0.0, atol=0.0)
torch.testing.assert_close(forced[0], control[0], rtol=0.0, atol=0.0)

lead_time_days = (
    torch.arange(control.shape[0]) * common["dt_save"] * time_unit_hours / 24.0
)
print("state shape:", tuple(control.shape))
print(f"final interpreted lead time: {float(lead_time_days[-1]):.2f} days")

initial_speed = torch.linalg.vector_norm(control[0, ..., 1:], dim=-1)
initial_speed_rms = float(torch.sqrt(initial_speed.square().mean()))
initial_height_rms = float(
    torch.sqrt((control[0, ..., 0] - h_mean).square().mean())
)
advective_crossing_days = (
    domain_size / initial_speed_rms * time_unit_hours / 24.0
)
print(f"initial RMS wind ~= {initial_speed_rms * velocity_unit:.1f} m/s")
print(
    f"initial 90th-percentile wind ~= "
    f"{float(torch.quantile(initial_speed, 0.9)) * velocity_unit:.1f} m/s"
)
print(
    f"initial RMS equivalent-height anomaly ~= "
    f"{initial_height_rms * equivalent_depth_metres:.1f} m"
)
print(f"Froude number ~= {initial_speed_rms / model_wave_speed:.2f}")
print(f"Rossby number at 600 km ~= {initial_speed_rms / (model_f0 * 16):.2f}")
print(f"drag e-folding time ~= {(1 / common['drag']) * time_unit_hours / 24:.1f} days")
print(f"advective crossing time ~= {advective_crossing_days:.1f} days")
print(
    f"rollout spans ~= {float(lead_time_days[-1]) / advective_crossing_days:.1f} "
    f"advective and {float(lead_time_days[-1]) / gravity_crossing_days:.1f} "
    "gravity-wave crossings"
)


## Long-rollout diagnostics

The simulator records the exact discrete energy used here,

$$E=\left\langle\tfrac12 h|\mathbf{u}|^2+\tfrac12g(h-H)^2\right\rangle.$$

For every saved transition, its change closes as $\Delta E=\Delta E_{\mathrm{RK4}}+\Delta E_{\mathrm{hyp}}+\Delta E_F$. The separate positive viscosity and drag work estimates diagnose the forcing scale but are already contained in the RK4 term. The plots also show the forced-minus-control separation and the effective stochastic diffusion rate.


In [ ]:
budget_index = {name: index for index, name in enumerate(simulator.energy_budget_names)}
control_budget = runs["none"]["energy_budget"][0]
forced_budget = runs["vortical"]["energy_budget"][0]

total_change = forced_budget[1:, 0] - forced_budget[:-1, 0]
operator_change = forced_budget[:-1, 1:4].sum(dim=-1)
torch.testing.assert_close(total_change, operator_change, rtol=2e-4, atol=2e-7)
cumulative_operator_change = torch.cat(
    (torch.zeros(1, 3), forced_budget[:-1, 1:4].cumsum(dim=0))
)

difference = forced - control
separation = torch.sqrt(
    (
        difference[..., 1:].square().sum(dim=-1)
        + (g / h_mean) * difference[..., 0].square()
    ).mean(dim=(1, 2))
)

forcing_impulse = runs["vortical"]["additional_input_fields"][0]
torch.testing.assert_close(
    forcing_impulse[-1], torch.zeros_like(forcing_impulse[-1])
)

diagnosed_loss_rate = (
    forced_budget[:-1, budget_index["viscous_dissipation_estimate"]]
    + forced_budget[:-1, budget_index["drag_dissipation_estimate"]]
    - forced_budget[:-1, budget_index["hyperviscous_energy_change"]]
) / common["dt_save"]
effective_rate = forced_budget[
    :-1, budget_index["effective_forcing_energy_rate"]
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
axes[0, 0].plot(lead_time_days, control_budget[:, 0], label="control")
axes[0, 0].plot(lead_time_days, forced_budget[:, 0], label="forced")
axes[0, 0].set_ylabel("exact specific energy")
axes[0, 0].legend()
for column, label in enumerate(("RK4", "hyperviscosity", "forcing")):
    axes[0, 1].plot(lead_time_days, cumulative_operator_change[:, column], label=label)
axes[0, 1].set_ylabel("cumulative exact energy change")
axes[0, 1].legend(fontsize=8)
axes[1, 0].plot(lead_time_days, separation)
axes[1, 0].set_ylabel("energy-weighted separation")
axes[1, 1].plot(lead_time_days[:-1], effective_rate, label="forcing target")
axes[1, 1].plot(
    lead_time_days[:-1],
    common["forcing_backscatter_fraction"] * diagnosed_loss_rate,
    "--",
    label="fraction of diagnosed loss",
)
axes[1, 1].set_ylabel("energy diffusion rate")
axes[1, 1].legend(fontsize=8)
for axis in axes:
    for panel in axis:
        panel.set_xlabel("interpreted lead time (days)")
        panel.grid(alpha=0.25)
plt.show()


In [ ]:
def periodic_vorticity(velocity, Lx, Ly):
    """Compute relative vorticity with periodic spectral derivatives."""
    nx, ny = velocity.shape[1:3]
    kx = 2.0 * torch.pi * torch.fft.fftfreq(nx, d=Lx / nx)
    ky = 2.0 * torch.pi * torch.fft.fftfreq(ny, d=Ly / ny)
    if nx % 2 == 0:
        kx[nx // 2] = 0.0
    if ny % 2 == 0:
        ky[ny // 2] = 0.0
    velocity_hat = torch.fft.fft2(velocity, dim=(1, 2))
    return torch.fft.ifft2(
        1j * kx[None, :, None] * velocity_hat[..., 1]
        - 1j * ky[None, None, :] * velocity_hat[..., 0],
        dim=(1, 2),
    ).real


control_fields = (
    control[-1, ..., 0] - h_mean,
    periodic_vorticity(control[..., 1:], domain_size, domain_size)[-1],
)
forced_fields = (
    forced[-1, ..., 0] - h_mean,
    periodic_vorticity(forced[..., 1:], domain_size, domain_size)[-1],
)
field_names = (r"height anomaly $h-H$", r"vorticity $\zeta$")

fig, axes = plt.subplots(2, 3, figsize=(11, 7), constrained_layout=True)
for row, (control_field, forced_field, field_name) in enumerate(
    zip(control_fields, forced_fields, field_names, strict=True)
):
    field_difference = forced_field - control_field
    shared_limit = max(
        float(control_field.abs().max()), float(forced_field.abs().max()), 1e-8
    )
    difference_limit = max(float(field_difference.abs().max()), 1e-8)
    for column, (field, title, limit) in enumerate(
        (
            (control_field, "control", shared_limit),
            (forced_field, "forced", shared_limit),
            (field_difference, "forced - control", difference_limit),
        )
    ):
        image = axes[row, column].imshow(
            field.T, origin="lower", cmap="RdBu_r", vmin=-limit, vmax=limit
        )
        axes[row, column].set_title(f"{title}: {field_name}")
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])
        fig.colorbar(image, ax=axes[row, column], shrink=0.75)
fig.suptitle(f"Final state at {float(lead_time_days[-1]):.1f} interpreted days")
plt.show()


In [ ]:
video_directory = Path("/tmp/autocast/outputs/swe_medium_range")
video_directory.mkdir(parents=True, exist_ok=True)


def display_velocity_vorticity(velocity, times, *, title, save_path, stride=4):
    """Display and save velocity and vorticity with global scales."""
    velocity = velocity.detach().cpu()
    times = torch.as_tensor(times).detach().cpu()
    speed = torch.linalg.vector_norm(velocity, dim=-1)
    vorticity = periodic_vorticity(
        velocity, domain_size, domain_size
    )
    speed_limit = max(float(speed.max()), 1e-8)
    vorticity_limit = max(float(vorticity.abs().max()), 1e-8)
    sampled_velocity = velocity[:, ::stride, ::stride]
    sampled_speed = torch.linalg.vector_norm(sampled_velocity, dim=-1)
    arrow_reference_speed = max(
        float(torch.quantile(sampled_speed, 0.9)), 1e-8
    )
    x = torch.arange(0, velocity.shape[1], stride)
    y = torch.arange(0, velocity.shape[2], stride)
    quiver_x, quiver_y = torch.meshgrid(x, y, indexing="xy")

    figure, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
    speed_image = axes[0].imshow(
        speed[0].T, origin="lower", cmap="magma", vmin=0, vmax=speed_limit
    )
    arrows = axes[0].quiver(
        quiver_x,
        quiver_y,
        sampled_velocity[0, ..., 0].T,
        sampled_velocity[0, ..., 1].T,
        color="white",
        edgecolor="black",
        linewidth=0.3,
        pivot="mid",
        scale=arrow_reference_speed / (0.6 * stride),
        scale_units="xy",
        angles="xy",
    )
    axes[0].set_title("speed and velocity")
    figure.colorbar(speed_image, ax=axes[0], label="speed")
    vorticity_image = axes[1].imshow(
        vorticity[0].T,
        origin="lower",
        cmap="RdBu_r",
        vmin=-vorticity_limit,
        vmax=vorticity_limit,
    )
    axes[1].set_title(r"relative vorticity $\partial_xv-\partial_yu$")
    figure.colorbar(vorticity_image, ax=axes[1], label="vorticity")
    time_title = figure.suptitle("")

    def update(frame):
        speed_image.set_data(speed[frame].T)
        vorticity_image.set_data(vorticity[frame].T)
        arrows.set_UVC(
            sampled_velocity[frame, ..., 0].T,
            sampled_velocity[frame, ..., 1].T,
        )
        time_title.set_text(f"{title}: lead {float(times[frame]):.1f} days")
        return speed_image, arrows, vorticity_image, time_title

    animation = FuncAnimation(
        figure, update, frames=velocity.shape[0], interval=200
    )
    animation.save(save_path, fps=5, dpi=120)
    display(HTML(animation.to_jshtml()))
    plt.close(figure)


comparison_animation = plot_spatiotemporal_video(
    true=forced.unsqueeze(0),
    pred=control.unsqueeze(0),
    batch_idx=0,
    fps=5,
    title="random-PV initialization: vortical forcing vs control",
    true_label="Forced",
    pred_label="Control",
    channel_names=["h", "u", "v"],
    colorbar_mode="column",
    preserve_aspect=True,
    save_path=str(video_directory / "random_pv_vs_control_15_day.mp4"),
)
display(HTML(comparison_animation.to_jshtml()))

forcing_animation = plot_spatiotemporal_video(
    true=forcing_impulse[:-1].unsqueeze(0),
    batch_idx=0,
    fps=5,
    title="realized forcing impulse over each saved interval",
    true_label="Forcing impulse",
    channel_names=["dh", "du", "dv"],
    preserve_aspect=True,
    save_path=str(video_directory / "random_pv_forcing_impulse_15_day.mp4"),
)
display(HTML(forcing_animation.to_jshtml()))

display_velocity_vorticity(
    forced[..., 1:],
    lead_time_days,
    title="forced random-PV flow",
    save_path=video_directory / "random_pv_velocity_vorticity_15_day.mp4",
)
print(f"Videos saved under {video_directory}")


## Useful variants

- Set `forcing_type="none"` for deterministic predictability and numerical-stability tests.
- Set `forcing_backscatter_fraction=0` and a nonzero `forcing_energy_rate` for fixed forcing independent of the flow.
- `vortical` is the closest option to spectral stochastic backscatter; `pv_balanced` is an experimental joint height--velocity construction.
- Use `initial_condition="random"` for the original mixed state, `"balanced_random_pv"` for distributed eddies, or `"balanced_double_jet"` for a controlled shear-flow benchmark.
- For random-PV dataset diversity, put `initial_wavenumber` and `initial_bandwidth` ranges in `parameters_range`; the sampled values are returned in `constant_scalars`.
- Supply any valid `[nx, ny, 3]` state with `initial_condition="restart"` to branch paired forecasts.
- Compare forcing modes 4 and 8 to move from roughly twice the deformation radius to one deformation radius.
- Set `nx=ny=128` while keeping `Lx=Ly=64` for better scale separation.
- For a ten-day paired forecast, use `T=222`, `dt_save=6`, and `skip_nt=0`.
- For a fifteen-day unpaired dataset, use `T=348`, `dt_save=6`, and `skip_nt=3`.

PlanetSWE-like deterministic travelling height forcing should remain a separate forcing mechanism from the stochastic OU input because the two represent different physical questions.
